In [ ]:
# transformers がすでにインストールされているかを確認し、
# 未インストールの場合のみインストールする
import importlib.util
import sys
import subprocess

# importlib.util.find_spec("transformers") は、
# Python環境内に transformers パッケージが存在するかを確認する
# None の場合は未インストール、None でなければインストール済み
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "transformers[ja,sentencepiece,torch]"]
    )
else:
    print("transformers はすでにインストールされています。")


In [3]:
import os
import warnings

# Hugging Face Hub からの認証なしリクエストに関する警告など、
# 実行結果に不要な警告を表示しないようにする。
warnings.filterwarnings("ignore")

# Hugging Face Hub の進捗バーを無効化する。
# モデルの重みダウンロード時に表示される tqdm の progress bar を抑制する。
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Transformers 側のログ出力を error のみに制限する。
# これにより、モデル読み込み時の warning / info レベルの出力を抑える。
from transformers.utils import logging

logging.set_verbosity_error()

from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 事前学習済み・ファインチューニング済みモデルによるテキスト分類を行う。
#
# pipeline は、トークナイズ・モデルへの入力・推論・出力整形をまとめて行う高水準API。
# ここでは感情極性分類、つまり文章がポジティブかネガティブかを判定するモデルを使う。
text_classification_pipeline = pipeline(
    # text-classification を明示することで、
    # この pipeline が文章分類タスク用であることをコード上から分かりやすくする。
    "text-classification",
    # llm-book/bert-base-japanese-v3-marc_ja は、
    # 日本語BERTを日本語レビュー分類データセットでファインチューニングしたモデル。
    #
    # BERTはTransformer Encoderをベースにしたモデルであり、
    # 文中の各トークンがSelf-Attentionによって他のトークンとの関係を考慮しながら表現される。
    #
    # テキスト分類では、文章全体の意味を表すベクトルを分類層に入力し、
    # positive / negative などのラベルごとのスコアを計算する。
    model="llm-book/bert-base-japanese-v3-marc_ja",
)

# ポジティブ寄りの文章を用意する。
# 「感動する」という語が含まれており、文全体として肯定的な評価を表している。
positive_text = "世界には言葉がわからなくても感動する音楽がある。"

# ネガティブ寄りの文章を用意する。
# 「ひどい」という語が含まれており、文全体として否定的な評価を表している。
negative_text = "世界には言葉がでないほどひどい音楽がある。"

# 複数の文章をリストにまとめて pipeline に渡す。
#
# 1文ずつ推論することもできるが、複数文をまとめて渡すことで、
# 入力と出力の対応関係を管理しやすくなる。
texts = [
    positive_text,
    negative_text,
]

# texts の各文章について感情極性を予測する。
#
# 内部では以下の処理が行われる。
#
# 1. 文章をトークンに分割する
# 2. トークンをID列に変換する
# 3. BERTに入力してSelf-Attentionにより文脈表現を得る
# 4. 分類層で各ラベルのスコアを計算する
# 5. 最も確率の高いラベルとそのスコアを返す
results = text_classification_pipeline(texts)

# 余計な説明文を出さず、分類結果だけを表示する。
for result in results:
    print(result)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20540.78it/s]


{'label': 'positive', 'score': 0.9993619322776794}
{'label': 'negative', 'score': 0.9636250138282776}
